# 1. Caricamento del dataset

In questo notebook viene valutato il classificatore **Decision Tree** progettato manualmente
nel Task 2 (notebook `02_manual_classifiers_2`), applicandolo questa volta al dataset
`training.csv` (50.000 osservazioni) anziché al solo `manuale.csv`.

L'obiettivo del Task 4 è verificare come si comporta il classificatore costruito a mano su una
quantità di dati molto più ampia e rappresentativa, e tentare di ottimizzarne le prestazioni.

In [1]:
import pandas as pd

training_df = pd.read_csv("../data/training.csv")

training_df.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,0,1,1,25,1,0,0,1,1,1,...,0,2,0,0,0,0,7,5,8,0
1,0,1,1,29,0,0,0,1,1,1,...,0,2,0,0,0,0,10,5,6,0
2,0,0,1,25,0,0,0,1,0,1,...,0,2,0,0,0,1,12,4,6,0
3,1,0,1,28,0,0,0,0,0,0,...,0,2,0,0,0,0,8,6,5,1
4,0,1,1,29,1,0,0,1,0,1,...,0,3,5,0,0,1,6,4,8,0


# 2. Preparazione dei dati

Nel Task 2 la variabile BMI è stata discretizzata in categorie per costruire l'albero di decisione.
La stessa trasformazione deve essere applicata anche al dataset `training.csv`, in modo che il
classificatore possa essere utilizzato.

L'albero sviluppato utilizza due feature: **BMI_Category** (ottenuta per discretizzazione del BMI)
e **GenHlth** (già presente nel dataset come variabile ordinale, quindi non richiede trasformazioni).

In [2]:
# Discretizzazione del BMI

training_df["BMI_Category"] = pd.cut(
    training_df["BMI"],
    bins=[0, 25, 30, 100],
    labels=["Normal", "Overweight", "Obese"],
    right=False
)

training_df[["BMI", "BMI_Category"]]

,BMI,BMI_Category
0,25,Overweight
1,29,Overweight
2,25,Overweight
3,28,Overweight
4,29,Overweight
...,...,...
49995,26,Overweight
49996,26,Overweight
49997,23,Normal
49998,36,Obese


In [3]:
# Tabella di contingenza per BMI_Category

pd.crosstab(
    training_df["BMI_Category"],
    training_df["Diabetes_binary"]
)

Diabetes_binary,0,1
BMI_Category,,
Normal,13354,817
Overweight,16429,2087
Obese,13250,4063


La variabile `GenHlth` è già espressa come valore ordinale da 1 (salute ottima) a 5 (salute pessima),
quindi può essere utilizzata direttamente dall'albero senza ulteriori trasformazioni. Verifichiamo la
sua distribuzione rispetto al target.

In [4]:
# Tabella di contingenza per GenHlth

pd.crosstab(
    training_df["GenHlth"],
    training_df["Diabetes_binary"]
)

Diabetes_binary,0,1
GenHlth,,
1,8644,242
2,16215,1209
3,12270,2707
4,4456,1902
5,1448,907


# 3. Richiamo del classificatore Decision Tree

In questa sezione viene riportata la struttura dell'albero di decisione costruito manualmente
nel Task 2. L'albero classifica le osservazioni in base a due sole feature (BMI_Category e GenHlth),
secondo le seguenti regole:

1. **SE** BMI è *Obese* → diabetico (1)
2. **SE** BMI è *Overweight* **E** GenHlth ∈ {2, 4} → diabetico (1)
3. **SE** BMI è *Overweight* **E** GenHlth ∈ {1, 3} → non diabetico (0)
4. **SE** BMI è *Normal* → non diabetico (0)

Queste regole erano state ricavate dall'analisi dell'Information Gain sui 14 campioni di `manuale.csv`.

# 4. Applicazione del classificatore

L'albero viene tradotto nella stessa funzione Python definita nel Task 2 e applicato a tutte
le osservazioni del dataset `training.csv`.

In [5]:
def predict_decision_tree(row):
    # Nodo radice: BMI_Category
    if row["BMI_Category"] == "Obese":
        return 1                      # foglia pura -> diabetico

    elif row["BMI_Category"] == "Overweight":
        # split su GenHlth: {2,4} -> diabetico, {1,3} -> non diabetico
        if row["GenHlth"] in [2, 4]:
            return 1
        else:
            return 0

    else:  # BMI_Category == "Normal"
        return 0                      # Normal -> non diabetico

## Predizione del dataset

Il classificatore viene applicato a tutte le osservazioni del dataset di training tramite il
metodo `apply`, che esegue la funzione riga per riga.

In [6]:
training_df["Predicted"] = training_df.apply(
    predict_decision_tree,
    axis=1
)

training_df[
    [
        "Diabetes_binary",
        "Predicted"
    ]
]

,Diabetes_binary,Predicted
0,0,1
1,0,1
2,0,1
3,1,1
4,0,0
...,...,...
49995,0,1
49996,0,0
49997,1,0
49998,1,1


# 5. Valutazione delle prestazioni

Dopo aver applicato il classificatore Decision Tree al dataset di training, vengono calcolate
le metriche di performance, utilizzando le stesse definizioni adottate nel Task 2.

# 5.1 Costruzione della tabella dei risultati

In [7]:
results_df = pd.DataFrame({
    "Actual": training_df["Diabetes_binary"],
    "Predicted": training_df["Predicted"]
})

results_df.head(10)

,Actual,Predicted
0,0,1
1,0,1
2,0,1
3,1,1
4,0,0
5,0,0
6,0,0
7,0,1
8,1,0
9,0,1


# 5.2 Calcolo manuale della Confusion Matrix

In [8]:
tp = len(
    training_df[(training_df["Diabetes_binary"] == 1) & (training_df["Predicted"] == 1)]
)

tn = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted"] == 0)]
)

fp = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted"] == 1)]
)

fn = len(
    training_df[ (training_df["Diabetes_binary"] == 1) & (training_df["Predicted"] == 0)]
)

print("TP =", tp)
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)

# Visualizzazione della matrice

confusion_matrix_df = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)

confusion_matrix_df

TP = 5010
TN = 21690
FP = 21343
FN = 1957


,Predicted 0,Predicted 1
Actual 0,21690,21343
Actual 1,1957,5010


## Accuracy

L'accuracy misura la percentuale di osservazioni classificate correttamente dal modello.

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

In [9]:
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("Accuracy: " , accuracy)

Accuracy:  0.534


### Precision

La precision misura la percentuale di osservazioni classificate come positive che sono
effettivamente positive.

$$Precision = \frac{TP}{TP + FP}$$

In [10]:
precision = tp / (tp + fp)

print("Precision: ", precision)

Precision:  0.1901111827875384


### Recall

La recall misura la capacità del classificatore di individuare correttamente le osservazioni
positive (i diabetici).

$$Recall = \frac{TP}{TP + FN}$$

In [11]:
recall = tp / (tp + fn)

print("Recall: ", recall)

Recall:  0.719104349074207


### F1-Score

L'F1-Score rappresenta la media armonica tra precision e recall.

$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

In [12]:
f1 = 2 * precision * recall / (precision + recall)

print("F1-Score: ", f1)

F1-Score:  0.30072028811524604


# 6. Analisi critica dei risultati

Il classificatore Decision Tree progettato manualmente è stato applicato al dataset `training.csv`
(50.000 osservazioni), ottenendo prestazioni molto diverse rispetto a quelle ottenute sul piccolo
dataset `manuale.csv` (dove l'accuracy era dell'85.71%).

Sul dataset completo le prestazioni risultano nettamente inferiori. Questo calo è atteso e si spiega
con il fatto che le regole dell'albero erano state ricavate da soli 14 campioni: un insieme troppo
piccolo per catturare i pattern reali presenti in decine di migliaia di osservazioni. In particolare:

- La regola **"Obese → sempre diabetico"** non è realistica: nel dataset completo solo una minoranza
  dei soggetti obesi è effettivamente diabetica. Questa regola genera quindi un numero molto elevato
  di **falsi positivi**.
- La regola **"Overweight con GenHlth ∈ {2,4} → diabetico"** si basava su una coincidenza del piccolo
  campione (i valori pari di GenHlth) e non su una relazione monotona reale: nei dati veri la probabilità
  di diabete cresce in modo continuo all'aumentare di GenHlth, non "a salti".

L'elevato numero di falsi positivi spiega la bassa precision. La recall resta invece relativamente alta,
perché un modello che predice "diabetico" molto spesso tende comunque a intercettare molti dei veri positivi.

Questo risultato evidenzia il limite principale di un classificatore costruito a mano su pochissimi dati:
una **scarsa capacità di generalizzazione**. Nella sezione successiva si tenta di ottimizzare l'albero
ricalibrando le sue regole sulla base dei pattern osservati in un campione più ampio del dataset.

# 7. Tentativo di ottimizzazione del classificatore

Le regole utilizzate dall'albero erano state derivate dai soli 14 campioni di `manuale.csv`.
Per migliorarne le prestazioni, analizziamo i pattern reali su un campione più ampio e ridefiniamo
le regole di decisione in modo coerente con i dati osservati.

Come nel notebook del classificatore Naive Bayes, si estrae un campione stratificato di 1000
osservazioni dal dataset di training.

In [13]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    training_df,
    train_size=1000,
    stratify=training_df["Diabetes_binary"],
    random_state=42
)

sample_df.shape

(1000, 24)

In [14]:
# Elimino la colonna `Predicted` creata durante il Task 4

sample_df = sample_df.drop(
    columns=["Predicted"],
    errors="ignore"
)

sample_df.columns

Index(['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke',
       'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income',
       'Diabetes_binary', 'BMI_Category'],
      dtype='str')

## Analisi dei pattern reali nel campione

Per ridefinire le regole dell'albero, analizziamo come varia la percentuale di diabetici (classe 1)
in funzione delle due feature usate dall'albero: BMI_Category e GenHlth.

In [15]:
# Percentuale di diabetici per categoria di BMI

bmi_rates = sample_df.groupby("BMI_Category", observed=True)["Diabetes_binary"].mean() * 100

print("Percentuale di diabetici per BMI_Category:")
print(bmi_rates.round(1))

Percentuale di diabetici per BMI_Category:
BMI_Category
Normal         5.8
Overweight    11.2
Obese         23.8
Name: Diabetes_binary, dtype: float64


In [16]:
# Percentuale di diabetici per livello di GenHlth

genhlth_rates = sample_df.groupby("GenHlth")["Diabetes_binary"].mean() * 100

print("Percentuale di diabetici per GenHlth:")
print(genhlth_rates.round(1))

Percentuale di diabetici per GenHlth:
GenHlth
1     2.3
2     6.6
3    17.8
4    31.7
5    31.9
Name: Diabetes_binary, dtype: float64


### Osservazioni dai dati reali

Dall'analisi del campione emergono due pattern chiari, molto diversi dalle regole ricavate dai 14 campioni:

- La probabilità di diabete **cresce in modo monotono** con il BMI (Normal < Overweight < Obese), ma
  anche tra gli obesi i diabetici restano una minoranza: la regola "Obese → sempre diabetico" è quindi
  troppo aggressiva.
- La probabilità di diabete **cresce in modo monotono** con GenHlth: è molto bassa per GenHlth = 1-2 e
  diventa elevata per GenHlth = 4-5. Questo conferma che GenHlth va trattata con una **soglia crescente**,
  non con i valori pari/dispari.

Sulla base di questi pattern, ridefiniamo le regole dell'albero combinando BMI_Category e una soglia
su GenHlth, in modo che a categorie di BMI più alte corrisponda una soglia di GenHlth più bassa per
classificare come diabetico.

## Ottimizzazione tramite ridefinizione delle regole

Il nuovo albero ottimizzato adotta le seguenti regole, calibrate sui pattern reali:

- **Obese** → diabetico se GenHlth ≥ 3 (la categoria a più alto rischio richiede una soglia più bassa)
- **Overweight** → diabetico se GenHlth ≥ 4
- **Normal** → diabetico solo se GenHlth ≥ 5 (la categoria a più basso rischio richiede la soglia più alta)

In questo modo la decisione tiene conto sia del peso corporeo sia dello stato di salute percepito,
in modo coerente con l'andamento crescente del rischio osservato nei dati.

In [17]:
def predict_decision_tree_v2(row):
    bmi = row["BMI_Category"]
    g = row["GenHlth"]

    if bmi == "Obese":
        return 1 if g >= 3 else 0
    elif bmi == "Overweight":
        return 1 if g >= 4 else 0
    else:  # Normal
        return 1 if g >= 5 else 0

In [18]:
predictions_v2 = training_df.apply(
    predict_decision_tree_v2,
    axis=1
)

training_df["Predicted_v2"] = predictions_v2

predictions_v2.value_counts()

0    36075
1    13925
Name: count, dtype: int64

# Costruzione della nuova tabella dei risultati

In [19]:
results_v2 = pd.DataFrame({
    "Actual": training_df["Diabetes_binary"],
    "Predicted": predictions_v2
})

results_v2.head(10)

,Actual,Predicted
0,0,0
1,0,0
2,0,0
3,1,0
4,0,0
5,0,0
6,0,0
7,0,0
8,1,1
9,0,0


# Calcolo manuale della nuova Confusion Matrix

In [20]:
tp_v2 = len(
    training_df[(training_df["Diabetes_binary"] == 1) & (training_df["Predicted_v2"] == 1)]
)

tn_v2 = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted_v2"] == 0)]
)

fp_v2 = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted_v2"] == 1)]
)

fn_v2 = len(
    training_df[ (training_df["Diabetes_binary"] == 1) & (training_df["Predicted_v2"] == 0)]
)

print("TP =", tp_v2)
print("TN =", tn_v2)
print("FP =", fp_v2)
print("FN =", fn_v2)

# Visualizzazione della matrice

confusion_matrix_v2_df = pd.DataFrame(
    [
        [tn_v2, fp_v2],
        [fn_v2, tp_v2]
    ],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)

confusion_matrix_v2_df

TP = 4197
TN = 33305
FP = 9728
FN = 2770


,Predicted 0,Predicted 1
Actual 0,33305,9728
Actual 1,2770,4197


# Ricalcolo Accuracy

In [21]:
accuracy_v2 = (tp_v2 + tn_v2) / (tp_v2 + tn_v2 + fp_v2 + fn_v2)

print("Accuracy Ricalcolata: " , accuracy_v2)

Accuracy Ricalcolata:  0.75004


# Ricalcolo Precision

In [22]:
precision_v2 = tp_v2 / (tp_v2 + fp_v2)

print("Precision Ricalcolata: ", precision_v2)

Precision Ricalcolata:  0.3014003590664273


# Ricalcolo Recall

In [23]:
recall_v2 = tp_v2 / (tp_v2 + fn_v2)

print("Recall Ricalcolata: ", recall_v2)

Recall Ricalcolata:  0.6024113678771351


# Ricalcolo F1-Score

In [24]:
f1_v2 = 2 * precision_v2 * recall_v2 / (precision_v2 + recall_v2)

print("F1-Score Ricalcolata: ", f1_v2)

F1-Score Ricalcolata:  0.4017805858701896


## Confronto diretto: Albero originale vs Albero ottimizzato

### Struttura delle regole

**ALBERO ORIGINALE (Task 2)** — regole derivate dai 14 campioni di `manuale.csv`:

SE BMI = Obese                          → diabetico (1)

SE BMI = Overweight E GenHlth ∈ {2,4}   → diabetico (1)

SE BMI = Overweight E GenHlth ∈ {1,3}   → non diabetico (0)

SE BMI = Normal                         → non diabetico (0)

**ALBERO OTTIMIZZATO (Task 4)** — regole ricalibrate sui pattern reali del campione:

SE BMI = Obese       E GenHlth ≥ 3   → diabetico (1)

SE BMI = Overweight  E GenHlth ≥ 4   → diabetico (1)

SE BMI = Normal      E GenHlth ≥ 5   → diabetico (1)

(in tutti gli altri casi              → non diabetico)

| Confusion Matrix | Albero originale | Albero ottimizzato |
|------------------|------------------|--------------------|
| Veri Positivi (TP)  | 5.010   | 4.197  |
| Veri Negativi (TN)  | 21.690  | 33.305 |
| Falsi Positivi (FP) | 21.343  | **9.728** |
| Falsi Negativi (FN) | 1.957   | 2.770  |

**1. Da "categoria" a "categoria + soglia".**
L'albero originale decideva guardando una sola feature alla volta (prima il BMI, e solo per gli Overweight anche GenHlth). L'albero ottimizzato combina **sempre entrambe** le feature: per ogni categoria di BMI applica una soglia su GenHlth. Questo permette decisioni molto più precise.

**2. La regola "Obese → sempre diabetico" è stata corretta.**
Era l'errore più grave dell'albero originale. Nei dati reali solo circa il 24% degli obesi è diabetico, quindi classificare *tutti* gli obesi come diabetici generava una valanga di falsi positivi. L'albero ottimizzato richiede che un obeso abbia anche GenHlth ≥ 3 per essere classificato come diabetico. Questa singola correzione è la principale responsabile del crollo dei falsi positivi: **da 21.343 a 9.728** (oltre 11.000 in meno).

**3. Da "pari/dispari" a "soglia crescente" su GenHlth.**
L'albero originale usava la regola GenHlth ∈ {2,4}, che era una pura coincidenza dei 14 campioni di `manuale.csv` (non un pattern reale). Nei dati veri la probabilità di diabete **cresce in modo monotono** con GenHlth (più la salute percepita è scarsa, più aumenta il rischio). L'albero ottimizzato rispetta questo andamento usando soglie del tipo `GenHlth ≥ k`.

**4. Soglie decrescenti al crescere del BMI.**
Nell'albero ottimizzato la soglia su GenHlth si abbassa man mano che il BMI aumenta (≥5 per Normal, ≥4 per Overweight, ≥3 per Obese). Questo riflette il fatto che il rischio dipende da *entrambi* i fattori: un BMI più alto "anticipa" il rischio, per cui serve un livello di compromissione della salute inferiore per classificare il soggetto come diabetico.

### Analisi critica dei risultati – Confronto tra modello originale e modello ottimizzato

Dopo aver applicato il classificatore Decision Tree manuale al dataset `training.csv`, sono state
calcolate le metriche di performance sia per il modello originale (basato sulle regole ricavate da
`manuale.csv`) sia per il modello ottimizzato (con regole ridefinite sui pattern osservati in un
campione più ampio di 1000 osservazioni).

| Metrica   | Modello originale | Modello ottimizzato |
|-----------|-------------------|---------------------|
| Accuracy  | 53.40%            | **75.00%**          |
| Precision | 19.01%            | **30.14%**          |
| Recall    | **71.91%**        | 60.24%              |
| F1-Score  | 30.07%            | **40.18%**          |

### Interpretazione del confronto

- **Accuracy**
  L'accuracy aumenta in modo molto marcato (dal 53.40% al 75.00%). Le nuove regole, basate sui pattern
  reali dei dati, riducono drasticamente il numero di falsi positivi prodotti dalla regola
  "Obese → sempre diabetico".

- **Precision**
  La precision migliora (dal 19.01% al 30.14%). Il modello ottimizzato commette molti meno falsi positivi:
  classifica come diabetici solo i soggetti che combinano un BMI elevato con uno stato di salute percepito
  peggiore.

- **Recall**
  La recall diminuisce (dal 71.91% al 60.24%). Il modello diventa più "conservativo": individua un po' meno
  diabetici reali, ma con molta più affidabilità. È il classico compromesso (trade-off) tra precision e recall.

- **F1-Score**
  L'F1-Score, che bilancia precision e recall, migliora in modo netto (dal 30.07% al 40.18%), segno che il
  guadagno complessivo è positivo nonostante il calo della recall.

### Considerazioni complessive

L'ottimizzazione ha prodotto un modello:

- **molto più accurato** nel complesso,
- **più preciso** (molti meno falsi positivi),
- **leggermente meno sensibile** (individua qualche caso di diabete in meno).

La differenza più importante rispetto al modello originale è la gestione del trade-off: il primo albero
era "ingenuo" e classificava troppo spesso come diabetico (alta recall ma bassissima precision e accuracy),
mentre il modello ottimizzato raggiunge un equilibrio molto migliore.

### Conclusione

Questo confronto mostra concretamente il limite di un classificatore costruito a mano su pochissimi dati:
le regole derivate da 14 campioni non generalizzano. Ricalibrando le stesse due feature (BMI e GenHlth)
sui pattern osservati in un campione rappresentativo, è stato possibile più che recuperare le prestazioni,
pur restando nei limiti di un modello a regole semplici.

Va comunque sottolineato che, anche dopo l'ottimizzazione, le prestazioni restano modeste in termini
assoluti: un albero a due sole feature non può competere con un classificatore addestrato automaticamente
su tutte le variabili disponibili. Questo motiva il passaggio al Task 5, in cui verranno addestrati tramite
Scikit-Learn classificatori più complessi, in grado di sfruttare l'intero insieme delle feature e di
ottimizzare automaticamente la struttura del modello.

Questa analisi costituisce, insieme a quella del classificatore Naive Bayes sviluppato dal compagno di
gruppo, la base per il confronto tra i due classificatori manuali e i modelli automatici del Task 5.